# Demo Mức 1 — reference-v1

Chạy từng ô bằng Shift+Enter. Notebook nạp một checkpoint GAT+CLIP seed 42,
suy luận CPU trên một ảnh validation và chấm bằng evaluator mới.
Đổi `CASE_ID` để xem một trong 10 case đã chuẩn bị (hit01–hit05 / miss01–miss05).
Các hit/miss là theo nhãn benchmark; nhận xét ảnh vẫn cần nhóm xác nhận.
Không train hoặc thay checkpoint. Một ảnh không đại diện chất lượng toàn tập.

In [ ]:
from pathlib import Path
import sys
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/models/gat.py').exists())
sys.path.insert(0, str(ROOT))


In [ ]:
import torch
from IPython.display import display
from PIL import Image
from scripts.evaluate_level1 import load_model
from scripts.extract_level1_examples import predict_graph
from scripts.train_mlp import make_dataset
from src.evaluation.reference_predcls import evaluate_reference_image
from src.knowledge.transe.core import sha256
torch.set_num_threads(1)
CASE_ID = 'miss03'  # thử hit01 (đúng), miss03 (hạng thấp), miss05 (nhiều GT cùng cặp).
CASE_ROOT = ROOT / 'reports/level1/submission/cases'
manifest = json.loads((CASE_ROOT / 'cases.json').read_text())
case = next(c for rows in manifest['cases'].values() for c in rows if c['case_id'] == CASE_ID)
checkpoint_path = ROOT / manifest['checkpoint']
assert sha256(checkpoint_path) == manifest['checkpoint_sha256']
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
config = checkpoint['config']
DATA = ROOT / 'data/raw/vg150'
for filename, digest in config['data_sha256'].items():
    assert sha256(DATA / filename) == digest
CACHE = ROOT / 'data/cache/vg150-clip-vit-b32-v1'
assert sha256(CACHE / 'manifest.json') == config['visual_cache_manifest_sha256']
dataset = make_dataset(DATA, 'val', config['box_policy'], CACHE)
graph = dataset[case['validation_index']]
assert str(graph.image_id) == case['image_id']
model = load_model(checkpoint).eval()
vocab = json.loads((DATA / 'VG-SGG-dicts-with-attri.json').read_text())
predicates = {v:k for k,v in vocab['predicate_to_idx'].items()}
print('Ảnh', graph.image_id, '| node:', graph.num_nodes, '| cặp ứng viên:', graph.edge_label_index.shape[1])
print('Đặc trưng mỗi node:', graph.x.shape[1], '= 5 hình học + 512 CLIP')

## Xem ảnh và cặp đối tượng

Đỏ = subject; xanh = object. Hộp/nhãn được cho sẵn. Quan hệ là thứ cần dự đoán.
Panel dưới ghi cả GT và dự đoán đã lưu để tiện giải thích. Đây là ảnh minh họa,
không được đưa vào forward của model dưới dạng chứa chữ/GT.

In [ ]:
display(Image.open(CASE_ROOT / case['panel']))
print('GT:', case['subject_label'], '→', case['gt_predicate'], '→', case['object_label'])
print('GT annotation row:', case['gt_annotation_row'])
gt_pair_mask = (graph.gt_relation_index[0] == case['subject_node']) & (graph.gt_relation_index[1] == case['object_node'])
print('Tất cả annotation GT của cặp này:', [predicates[int(p)] for p in graph.gt_predicate[gt_pair_mask]])
print('Hit/miss đang xét annotation đã chọn, không đại diện mọi quan hệ của cặp.')


In [ ]:
with torch.inference_mode():
    logits = predict_graph(model, config['model'], graph, torch.device('cpu'))
result = evaluate_reference_image(logits, graph.edge_label_index, graph.gt_relation_index,
                                  graph.gt_predicate, graph.boxes, graph.object_labels)
pairs = graph.edge_label_index
pair_id = torch.where((pairs[0] == case['subject_node']) & (pairs[1] == case['object_node']))[0].item()
scores, labels = logits.softmax(-1)[:, 1:].max(-1)
ranking = scores.sort(descending=True).indices
rank = int(torch.where(ranking == pair_id)[0]) + 1
prediction = predicates[int(labels[pair_id]) + 1]
assert prediction == case['same_pair_prediction'] and rank == case['same_pair_rank']
print('Suy luận lại:', prediction, '| hạng cặp:', rank)
for k in (50, 100):
    print(f'R@{k} ảnh: {result["hits"][k]}/{result["support"]} = {result["hits"][k] / result["support"]:.2%}')
print('Matching GT ở hạng:', case['first_reference_match_rank'])
print('Case thuộc nhóm:', case['kind'])
print('Lưu ý: reference matching dùng nhãn + IoU; đôi khi dự đoán từ node trùng/overlap cũng match GT.')

## Bổ sung: cho phép nhiều quan hệ trên cùng cặp

Dùng chính logits vừa dự đoán, đưa tất cả predicate foreground vào danh sách xếp hạng toàn ảnh.
Mỗi bộ ba chiếm một vị trí. Đổi `CASE_ID = 'miss05'` rồi chạy lại để xem cặp girl → hair có cả GT has và with.
Hạng dưới đây thuộc đúng cặp node đang xem; matching reference còn cho phép hộp/nhãn tương đương qua IoU.


In [ ]:
import numpy as np
ng_result = evaluate_reference_image(logits, pairs, graph.gt_relation_index, graph.gt_predicate,
                                     graph.boxes, graph.object_labels, graph_constraint=False)
ng_scores = logits.softmax(-1)[:, 1:][ranking].numpy().astype(np.float64)
flat_order = np.argsort(-ng_scores.ravel())
ng_pair_rows, ng_predicate_cols = np.unravel_index(flat_order, ng_scores.shape)
ng_pair_ids = ranking.numpy()[ng_pair_rows]
for predicate in sorted(set(graph.gt_predicate[gt_pair_mask].tolist())):
    positions = np.flatnonzero((ng_pair_ids == pair_id) & (ng_predicate_cols + 1 == predicate))
    ng_rank = int(positions[0]) + 1
    print(f"GT {predicates[predicate]}: hạng NG của đúng cặp = {ng_rank}; trong top 50: {ng_rank <= 50}")
for k in (50, 100):
    print(f"Ảnh này — R@{k}: {result['hits'][k]}/{result['support']}; "
          f"ng-R@{k}: {ng_result['hits'][k]}/{ng_result['support']}")
print("Hai cách chấm khác nhau; điểm NG cao hơn không có nghĩa là mô hình đã được cải thiện.")


## Giải thích và kết thúc demo

- Nếu predicate đúng nhưng hạng >50: lỗi ranking tại R@50, có thể được tìm lại ở R@100.
- Nếu predicate khác: xem có phải nhầm quan hệ, nhãn gần nghĩa, hoặc GT/hộp khó xác nhận.
- R ở trên chỉ là recall **một ảnh**. mR của toàn tập trung bình recall mỗi lớp qua các ảnh chứa lớp đó, rồi trung bình đủ 50 lớp.
- Các checkpoint vẫn được chọn theo metric legacy; reference-v1 là chấm lại, không chọn lại.

Mở `reports/level1/submission/cases/analysis.md` để xem nhận xét AI, và điền
`human-review.csv` sau khi chính bạn kiểm tra. Không suy ra chất lượng toàn tập từ case này.